In [ ]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    """Custom LoRA block: adds a low-rank update (B @ A) * (alpha / r)
    parallel to a frozen linear projection (e.g. Wq, Wk, Wv)."""

    def __init__(self, in_features, out_features, r=8, alpha=16):
        super().__init__()
        self.r = r
        self.scaling = alpha / r

        # A: down-projection, B: up-projection. B starts at zero so the
        # adapter contributes nothing until training moves it.
        self.A = nn.Parameter(torch.randn(in_features, r) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, out_features))

    def forward(self, x):
        return (x @ self.A @ self.B) * self.scaling


class LoRALinear(nn.Module):
    """Wraps an existing frozen nn.Linear with a LoRA adapter."""

    def __init__(self, base_linear: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False                       # freeze base weights

        self.lora = LoRALayer(base_linear.in_features, base_linear.out_features, r, alpha)

    def forward(self, x):
        return self.base(x) + self.lora(x)


def apply_lora_to_model(model, target_modules=("q_proj", "v_proj"), r=8, alpha=16):
    """Walks a model and replaces attention projections with LoRA-wrapped versions."""
    for name, module in model.named_children():
        if name in target_modules and isinstance(module, nn.Linear):
            setattr(model, name, LoRALinear(module, r=r, alpha=alpha))
        else:
            apply_lora_to_model(module, target_modules, r, alpha)
    return model


if __name__ == "__main__":
    class ToyAttention(nn.Module):
        def __init__(self, d_model=32):
            super().__init__()
            self.q_proj = nn.Linear(d_model, d_model)
            self.v_proj = nn.Linear(d_model, d_model)

        def forward(self, x):
            return self.q_proj(x) + self.v_proj(x)

    model = ToyAttention()
    model = apply_lora_to_model(model, r=4, alpha=8)

    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    frozen = [n for n, p in model.named_parameters() if not p.requires_grad]
    print("Trainable (LoRA) params:", trainable)
    print("Frozen (base) params:", frozen)

    x = torch.randn(2, 32)
    out = model(x)
    loss = out.sum()
    loss.backward()
    print("Forward/backward OK, output shape:", out.shape)

---
## Task 7: LoRA Matrix Projection Fine-Tuning